In [48]:
import numpy as np
import pandas as pd
from typing import Optional

In [49]:
# Load in movies dataset from parent directory

# movies = pd.read_csv('../ml-32m/movies.csv')
# ratings = pd.read_csv('../ml-32m/ratings.csv')
# movies.head()

# Code for 100k dataset:
ratings = pd.read_csv(
    "../ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "../ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

# Name movies columns
movies = movies[[0, 1]]
movies.columns = ["movie_id", "title"]

print(movies.head())

   movie_id              title
0         1   Toy Story (1995)
1         2   GoldenEye (1995)
2         3  Four Rooms (1995)
3         4  Get Shorty (1995)
4         5     Copycat (1995)


In [31]:
ratings.head()


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [32]:
# Use this if files are uploaded on google drive:

'''
from google.colab import drive
drive.mount('/content/drive')

movies = pd.read_csv('../ml-32m/movies.csv')
ratings = pd.read_csv('../ml-32m/ratings.csv')
'''

"\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nmovies = pd.read_csv('../ml-32m/movies.csv')\nratings = pd.read_csv('../ml-32m/ratings.csv')\n"

In [50]:
# Ratings dataset is too big! Reduce to 400,000 rows (~2500 users)

ratings = ratings.sample(n=10000, random_state=42)

In [51]:
# Merge datasets on movie ID

df = ratings.merge(movies, on='movie_id')


# Create user-movie matrix

user_movie_matrix = df.pivot_table(
    index="user_id",
    columns="title",
    values="rating"
)

In [35]:
# Simple recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Sort correlations highest to lowest
    recommendations = corr_df.sort_values(by='correlation', ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [36]:
# recommend_movies('Love Actually (2003)')

In [37]:
# Slightly more complicated recommender function

def recommend_movies(movie_title, top_movies=10):

    # Get ratings for selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Find correlation between this movie and other
    similar_movies = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to dataframe
    corr_df = pd.DataFrame(similar_movies, columns=["correlation"])

    # Remove NaN values
    corr_df = corr_df.dropna()

    # Count number of ratings per movie
    rating_counts = df.groupby("title")["rating"].count()

    # Add rating counts
    corr_df["num_ratings"] = rating_counts

    # Filter out unpopular movies
    recommendations = corr_df[corr_df["num_ratings"] >= 30].sort_values(
        by="correlation",
        ascending=False)

    # Remove the movie itself
    recommendations = recommendations.drop(movie_title, errors="ignore")

    # Return top movies
    return recommendations.head(top_movies)


In [52]:
def recommend_movies(
    movie_title: str,
    user_movie_matrix: pd.DataFrame,
    ratings_df: pd.DataFrame,
    top_n: int = 10,
    min_ratings: int = 30,
    shrinkage: float = 30.0,
    precomputed_corr: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """
    Item-based recommender: returns top-N movies similar to `movie_title`.

    Args:
      movie_title: title present in `user_movie_matrix` columns.
      user_movie_matrix: pivot table index=user_id, columns=title, values=rating.
      ratings_df: original ratings DataFrame with columns ['user_id','title','rating'].
      top_n: number of recommendations to return.
      min_ratings: minimum number of ratings required to consider a movie.
      shrinkage: parameter for Bayesian shrinkage (higher -> more shrink).
      precomputed_corr: optional DataFrame of pairwise correlations (columns and index are titles).

    Returns:
      DataFrame with columns ['correlation', 'num_ratings', 'score'] sorted by `score` desc.
    """
    if movie_title not in user_movie_matrix.columns:
        raise ValueError(f"movie_title not found: {movie_title!r}")

    if precomputed_corr is None:
        movie_ratings = user_movie_matrix[movie_title]
        simil = user_movie_matrix.corrwith(movie_ratings)
        corr_df = pd.DataFrame(simil, columns=["correlation"])
    else:
        if movie_title not in precomputed_corr.columns:
            raise ValueError(f"movie_title not in precomputed_corr: {movie_title!r}")
        corr_df = pd.DataFrame(precomputed_corr[movie_title].copy())
        corr_df.columns = ["correlation"]

    corr_df = corr_df.dropna()

    # Count number of ratings per movie (align to corr_df index)
    counts = ratings_df.groupby("title")["rating"].count()
    corr_df["num_ratings"] = counts.reindex(corr_df.index).fillna(0).astype(int)

    # Shrink correlation toward 0 for low counts: score = corr * (n / (n + shrinkage))
    corr_df["score"] = corr_df["correlation"] * (corr_df["num_ratings"] / (corr_df["num_ratings"] + shrinkage))

    # Filter unpopular movies and remove the seed movie
    candidates = corr_df[corr_df["num_ratings"] >= min_ratings].drop(index=movie_title, errors="ignore")

    # Sort by score then correlation as tiebreaker
    result = candidates.sort_values(by=["score", "correlation"], ascending=False).head(top_n)

    return result

In [55]:
recommend_movies("Toy Story (1995)", user_movie_matrix=user_movie_matrix, ratings_df=df)

/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mark/OMDS/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,correlation,num_ratings,score
title,,,
Blade Runner (1982),1.000000,41,0.577465
Return of the Jedi (1983),0.866025,54,0.556731
Apollo 13 (1995),1.000000,32,0.516129
Pulp Fiction (1994),1.000000,32,0.516129
Chasing Amy (1997),0.866025,35,0.466321
Sense and Sensibility (1995),0.597614,34,0.317483
Dead Man Walking (1995),0.547723,32,0.282696
Star Trek: First Contact (1996),0.448276,31,0.227812
Twelve Monkeys (1995),0.327327,38,0.182918


In [40]:
# movie predictor

movie_corrs_df = user_movie_matrix.corr()

In [41]:
movie_corrs_df = movie_corrs_df.dropna()
movie_corrs_df.head()

title,1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",8 1/2 (1963),...,Wonderland (1997),"World of Apu, The (Apur Sansar) (1959)","Wrong Trousers, The (1993)",Wyatt Earp (1994),Year of the Horse (1997),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997)
title,,,,,,,,,,,,,,,,,,,,,


In [42]:
ratings = pd.read_csv(
    "../ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [43]:
ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
